# Notebook 01: Vector Fundamentals & Distance Metrics

Welcome to the first notebook in the **Embeddings from Scratch** series!

Before diving into neural networks or complex models like BERT and Word2Vec, we must master the **fundamental language of embeddings**: **Vectors and Geometric Spaces**.

### Learning Objectives:
1. Understand what a vector is geometrically and algebraically.
2. Implement vector norms ($L_1$, $L_2$) and unit normalization from scratch.
3. Compare distance & similarity metrics: **Euclidean Distance**, **Manhattan Distance**, **Dot Product**, and **Cosine Similarity**.
4. Learn the mathematical trick: Why normalized dot product equals cosine similarity.
5. Explore the **Curse of Dimensionality** and why high-dimensional vectors behave differently than 2D/3D vectors.


---
## 1. Setup & Environment


In [1]:
# Standard imports
import numpy as np
import matplotlib.pyplot as plt

# Set random seed for reproducibility
np.random.seed(42)
print("Libraries imported successfully! NumPy version:", np.__version__)


Libraries imported successfully! NumPy version: 2.0.2


---
## 2. What is a Vector?

Algebraically, a vector is an ordered sequence of numbers:
$$\mathbf{v} = [v_1, v_2, \dots, v_d] \in \mathbb{R}^d$$
where $d$ is the **dimensionality** of the vector.

Geometrically, a vector represents:
1. A **point** in a $d$-dimensional coordinate space.
2. An **arrow** originating from the origin $(0, 0, \dots, 0)$ pointing to that coordinate, characterized by:
   - **Magnitude** (length)
   - **Direction** (angle relative to coordinate axes)

In NLP and RAG:
* Each document, sentence, or token is mapped to a vector in $\mathbb{R}^d$ (e.g., $d = 384, 768, 1536$).
* Concepts with similar meanings are positioned with similar directions/coordinates.


In [2]:
# Representing two simple 2D vectors
v1 = np.array([3.0, 4.0])
v2 = np.array([4.0, 1.0])

print(f"Vector 1: {v1}, Shape: {v1.shape}")
print(f"Vector 2: {v2}, Shape: {v2.shape}")


Vector 1: [3. 4.], Shape: (2,)
Vector 2: [4. 1.], Shape: (2,)


---
## 3. Vector Norms (Measuring Vector Length)

How "long" is a vector? We measure this using **norms**.

### 3.1 Euclidean Norm ($L_2$ Norm)
The straight-line length from the origin to the point:
$$\|\mathbf{v}\|_2 = \sqrt{\sum_{i=1}^d v_i^2} = \sqrt{\mathbf{v} \cdot \mathbf{v}}$$

### 3.2 Manhattan Norm ($L_1$ Norm)
The sum of absolute coordinates (grid distance):
$$\|\mathbf{v}\|_1 = \sum_{i=1}^d |v_i|$$

### 3.3 Normalization (Unit Vectors)
A vector is a **unit vector** if its $L_2$ norm is $1$. To normalize any non-zero vector:
$$\hat{\mathbf{v}} = \frac{\mathbf{v}}{\|\mathbf{v}\|_2}$$


In [3]:
def l2_norm(v: np.ndarray) -> float:
    """Compute L2 norm from scratch without np.linalg.norm"""
    return np.sqrt(np.sum(v ** 2))

def l1_norm(v: np.ndarray) -> float:
    """Compute L1 norm from scratch"""
    return np.sum(np.abs(v))

def normalize_l2(v: np.ndarray) -> np.ndarray:
    """Normalize vector to unit length (L2 norm = 1)"""
    norm = l2_norm(v)
    if norm == 0:
        return v
    return v / norm

# Testing norms on v1 = [3, 4]
print(f"v1: {v1}")
print(f"L2 Norm (length): {l2_norm(v1)} (expected 5.0)")
print(f"L1 Norm: {l1_norm(v1)} (expected 7.0)")

v1_unit = normalize_l2(v1)
print(f"Normalized v1: {v1_unit}")
print(f"Length of normalized v1: {l2_norm(v1_unit):.4f}")


v1: [3. 4.]
L2 Norm (length): 5.0 (expected 5.0)
L1 Norm: 7.0 (expected 7.0)
Normalized v1: [0.6 0.8]
Length of normalized v1: 1.0000


---
## 4. Measuring Distance and Similarity

When searching for relevant text chunks in RAG, we must quantify how close a **query vector** $\mathbf{q}$ is to candidate **document vectors** $\mathbf{d}$.

### 4.1 Euclidean Distance ($L_2$ Distance)
Straight-line distance between two points:
$$D_{L_2}(\mathbf{u}, \mathbf{v}) = \sqrt{\sum_{i=1}^d (u_i - v_i)^2}$$
* **Range**: $[0, \infty)$
* **Meaning**: Lower is closer / more similar.

### 4.2 Manhattan Distance ($L_1$ Distance)
Grid-like distance:
$$D_{L_1}(\mathbf{u}, \mathbf{v}) = \sum_{i=1}^d |u_i - v_i|$$

### 4.3 Dot Product (Inner Product)
$$\mathbf{u} \cdot \mathbf{v} = \sum_{i=1}^d u_i v_i = \|\mathbf{u}\|_2 \|\mathbf{v}\|_2 \cos(\theta)$$
* Captures **both** angle $\theta$ and lengths $\|\mathbf{u}\|_2, \|\mathbf{v}\|_2$.
* Sensitive to vector magnitude (longer vectors produce larger dot products).

### 4.4 Cosine Similarity
Measures only the **angle $\theta$** between vectors, ignoring length:
$$\text{CosineSimilarity}(\mathbf{u}, \mathbf{v}) = \cos(\theta) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\|_2 \|\mathbf{v}\|_2}$$
* **Range**: $[-1, 1]$
  * $+1$: Exactly same direction ($0^\circ$)
  * $0$: Orthogonal / completely unrelated ($90^\circ$)
  * $-1$: Directly opposite direction ($180^\circ$)


In [4]:
def euclidean_distance(u: np.ndarray, v: np.ndarray) -> float:
    return np.sqrt(np.sum((u - v) ** 2))

def manhattan_distance(u: np.ndarray, v: np.ndarray) -> float:
    return np.sum(np.abs(u - v))

def dot_product(u: np.ndarray, v: np.ndarray) -> float:
    return np.sum(u * v)

def cosine_similarity(u: np.ndarray, v: np.ndarray) -> float:
    norm_u = l2_norm(u)
    norm_v = l2_norm(v)
    if norm_u == 0 or norm_v == 0:
        return 0.0
    return dot_product(u, v) / (norm_u * norm_v)

# Evaluate metrics between v1 and v2
print(f"Vectors: v1={v1}, v2={v2}")
print(f"Euclidean Distance: {euclidean_distance(v1, v2):.4f}")
print(f"Manhattan Distance: {manhattan_distance(v1, v2):.4f}")
print(f"Dot Product:        {dot_product(v1, v2):.4f}")
print(f"Cosine Similarity:  {cosine_similarity(v1, v2):.4f}")


Vectors: v1=[3. 4.], v2=[4. 1.]
Euclidean Distance: 3.1623
Manhattan Distance: 4.0000
Dot Product:        16.0000
Cosine Similarity:  0.7761


---
## 5. Critical Insight: Why Normalize Before Search?

Notice the mathematical connection between Euclidean Distance, Dot Product, and Cosine Similarity:

$$\|\mathbf{u} - \mathbf{v}\|^2_2 = \|\mathbf{u}\|^2_2 + \|\mathbf{v}\|^2_2 - 2 (\mathbf{u} \cdot \mathbf{v})$$

If both $\mathbf{u}$ and $\mathbf{v}$ are **unit normalized** ($\|\mathbf{u}\|_2 = 1$ and $\|\mathbf{v}\|_2 = 1$):
$$\|\mathbf{u} - \mathbf{v}\|^2_2 = 1 + 1 - 2 (\mathbf{u} \cdot \mathbf{v}) = 2 - 2 \cdot \text{CosineSimilarity}(\mathbf{u}, \mathbf{v})$$

And:
$$\text{CosineSimilarity}(\mathbf{u}, \mathbf{v}) = \mathbf{u} \cdot \mathbf{v}$$

> **Key Takeaway for RAG / Vector Databases**:
> If you $L_2$-normalize all embeddings before inserting them into a vector DB (Pinecone, Qdrant, Milvus, Chroma), **Cosine Similarity becomes an ordinary Dot Product**, and ranking by Dot Product is identical to ranking by Euclidean Distance!
> Dot products can be computed extremely fast using matrix multiplication (BLAS / GPUs).


In [ ]:
# Demonstration of the equivalence
u_norm = normalize_l2(v1)
v_norm = normalize_l2(v2)

cos_sim = cosine_similarity(v1, v2)
dot_prod_normalized = dot_product(u_norm, v_norm)
sq_euclidean = euclidean_distance(u_norm, v_norm) ** 2
expected_sq_euclidean = 2 - 2 * cos_sim

print(f"Cosine Similarity:                    {cos_sim:.6f}")
print(f"Dot Product of Normalized Vectors:    {dot_prod_normalized:.6f}")
print(f"Squared Euclidean Distance:           {sq_euclidean:.6f}")
print(f"Formula (2 - 2 * CosineSimilarity):   {expected_sq_euclidean:.6f}")


---
## 6. Visualizing Vector Spaces & Invariance

Why do we prefer Cosine Similarity over Euclidean Distance for text?

Consider two documents:
* **Doc A**: Short sentence about artificial intelligence.
* **Doc B**: Long article repeating the exact same points about artificial intelligence 10 times.

Because Doc B repeats words, its vector magnitude will be much larger.
- Euclidean distance will say Doc A and Doc B are **far apart**.
- Cosine similarity will say Doc A and Doc B have an angle of $0^\circ$ (**identical topic**).


In [ ]:
# Visualizing magnitude invariance
doc_short = np.array([1.0, 2.0])
doc_long  = np.array([4.0, 8.0])   # Same direction, 4x magnitude
doc_other = np.array([2.0, 0.5])   # Different direction

fig, ax = plt.subplots(figsize=(7, 7))
origin = np.array([0, 0])

ax.quiver(*origin, *doc_short, color='blue', scale=1, scale_units='xy', angles='xy', label=f'Short Doc: {doc_short}')
ax.quiver(*origin, *doc_long,  color='cyan', scale=1, scale_units='xy', angles='xy', label=f'Long Doc (same topic): {doc_long}')
ax.quiver(*origin, *doc_other, color='red',  scale=1, scale_units='xy', angles='xy', label=f'Different Topic: {doc_other}')

ax.set_xlim(-1, 10)
ax.set_ylim(-1, 10)
ax.set_xlabel("Dimension 1 (e.g., Tech keywords)")
ax.set_ylabel("Dimension 2 (e.g., Science keywords)")
ax.grid(True, linestyle="--", alpha=0.5)
ax.legend()
plt.title("Direction (Topic) vs. Magnitude (Length)")
plt.show()

print(f"Euclidean(doc_short, doc_long):  {euclidean_distance(doc_short, doc_long):.4f} (looks far!)")
print(f"Cosine(doc_short, doc_long):     {cosine_similarity(doc_short, doc_long):.4f} (identifies identical topic!)")


---
## 7. The Curse of Dimensionality: High-Dimensional Spaces

In 2D or 3D, two random vectors can point in any direction.
What happens when $d = 768$ (BERT) or $d = 1536$ (OpenAI)?

Let's run a simulation: Draw 1,000 random vectors in 2D vs. 768D and measure the distribution of angles (or cosine similarities) between them.


In [ ]:
dimensions = [2, 10, 100, 768]
num_pairs = 2000

fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)

for ax, d in zip(axes, dimensions):
    # Generate random vectors from Gaussian distribution
    vecs_a = np.random.randn(num_pairs, d)
    vecs_b = np.random.randn(num_pairs, d)
    
    # Normalize rows
    vecs_a = vecs_a / np.linalg.norm(vecs_a, axis=1, keepdims=True)
    vecs_b = vecs_b / np.linalg.norm(vecs_b, axis=1, keepdims=True)
    
    # Cosine similarities (dot products of normalized vectors)
    cos_sims = np.sum(vecs_a * vecs_b, axis=1)
    
    ax.hist(cos_sims, bins=50, color='purple', alpha=0.7, density=True)
    ax.set_title(f"Dimension d = {d}")
    ax.set_xlabel("Cosine Similarity")
    ax.set_xlim(-1, 1)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel("Probability Density")
plt.suptitle("Curse of Dimensionality: Distribution of Pairwise Cosine Similarities Between Random Vectors", fontsize=13)
plt.tight_layout()
plt.show()


### What Did We Just Observe?
* In **2D**, random vectors have cosine similarities spread across the entire range $[-1, 1]$.
* In **768D**, the cosine similarities **concentrate sharply around 0.0** (orthogonal).
* In high-dimensional spaces:
  1. Almost every random pair of vectors is nearly **orthogonal** ($90^\circ$).
  2. Any non-zero cosine similarity (e.g., $0.6$ or $0.7$) indicates strong, meaningful semantic alignment!

---
## Summary of Notebook 01
1. **Vectors** represent text in continuous geometric spaces.
2. **$L_2$ Normalization** projects all vectors onto a unit hypersphere.
3. **Cosine Similarity** measures semantic orientation regardless of document length.
4. **Normalized Dot Product = Cosine Similarity**, enabling lightning-fast GPU vector search in production RAG systems.
5. In high dimensions ($d \ge 384$), random vectors are almost purely orthogonal.

In **Notebook 02**, we will see how words were first converted into these dense vectors using **Word2Vec (CBOW & Skip-Gram)**!
